In [3]:
# Install into the active notebook kernel (keep LangChain packages mutually compatible)
%pip install -q -U cassio datasets pyasyncore langchain langchain-community langchain-classic langchain-openai langchain-groq langchain-huggingface openai tiktoken python-dotenv "pyarrow<21"

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
Note: you may need to restart the kernel to use updated packages.


In [ ]:
# LangChain components to use
from langchain_community.vectorstores import Cassandra
from langchain_classic.indexes.vectorstore import VectorStoreIndexWrapper
from langchain_openai import OpenAI, OpenAIEmbeddings

# Support for dataset retrieval with Hugging Face
from datasets import load_dataset


import cassio

2026-04-11 09:41:54.407973: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [5]:
from pypdf import PdfReader

In [5]:
# Store Astra credentials outside the notebook.
# Required: ASTRA_DB_APPLICATION_TOKEN and either ASTRA_DB_ID or ASTRA_DB_SECURE_BUNDLE_PATH.
# Optional: ASTRA_DB_KEYSPACE

In [18]:
import os
from pathlib import Path

try:
    from dotenv import load_dotenv
except ImportError:
    load_dotenv = None

if load_dotenv is not None:
    for candidate in (
        Path('.env'),
        Path('Langchain/.env'),
        Path('vectordatabase_langchain/.env'),
        Path('../.env'),
        Path('../vectordatabase_langchain/.env'),
    ):
        if candidate.exists():
            load_dotenv(candidate, override=False)

def _env_or_default(name: str, default: str = '') -> str:
    value = os.getenv(name, '')
    value = value.strip() if isinstance(value, str) else ''
    return value or default

ASTRA_DB_APPLICATION_TOKEN = _env_or_default('ASTRA_DB_APPLICATION_TOKEN', '')
ASTRA_DB_ID = _env_or_default('ASTRA_DB_ID', '7334a689-2bd0-4640-b910-4e0b9a76707b')
ASTRA_DB_KEYSPACE = _env_or_default('ASTRA_DB_KEYSPACE', 'default_keyspace')
ASTRA_DB_SECURE_BUNDLE_PATH = _env_or_default('ASTRA_DB_SECURE_BUNDLE_PATH', '')

if not ASTRA_DB_APPLICATION_TOKEN.startswith('AstraCS:'):
    raise ValueError('Invalid ASTRA_DB_APPLICATION_TOKEN. Use a token that starts with AstraCS:.')

if ASTRA_DB_SECURE_BUNDLE_PATH and not Path(ASTRA_DB_SECURE_BUNDLE_PATH).expanduser().is_file():
    raise FileNotFoundError(f'Secure bundle not found: {ASTRA_DB_SECURE_BUNDLE_PATH}')

print(f'Using Astra DB ID: {ASTRA_DB_ID}')
print(f'Using keyspace: {ASTRA_DB_KEYSPACE}')

Using Astra DB ID: 7334a689-2bd0-4640-b910-4e0b9a76707b
Using keyspace: default_keyspace


In [7]:
pdf_reader = PdfReader("../vectordatabase_langchain/budget_speech (1).pdf")

In [8]:
from typing_extensions import Concatenate
raw_text=''
for i, page in enumerate(pdf_reader.pages):
    text=page.extract_text()
    raw_text+=text

In [9]:
raw_text[:1000]

'GOVERNMENT OF INDIA\nBUDGET 2023-2024\nSPEECH\nOF\nNIRMALA SITHARAMAN\nMINISTER OF FINANCE\nFebruary 1,  2023CONTENTS \nPART-A \n Page No. \n\uf0b7 Introduction 1 \n\uf0b7 Achievements since 2014: Leaving no one behind 2 \n\uf0b7 Vision for Amrit Kaal – an empowered and inclusive economy 3 \n\uf0b7 Priorities of this Budget 5 \ni. Inclusive Development  \nii. Reaching the Last Mile \niii. Infrastructure and Investment \niv. Unleashing the Potential \nv. Green Growth \nvi. Youth Power  \nvii. Financial Sector \n \n \n \n \n \n \n \n \n \n\uf0b7 Fiscal Management \n24 \nPART B \n  \nIndirect Taxes 27 \n\uf0b7 Green Mobility  \n\uf0b7 Electronics   \n\uf0b7 Electrical   \n\uf0b7 Chemicals and Petrochemicals   \n\uf0b7 Marine products  \n\uf0b7 Lab Grown Diamonds  \n\uf0b7 Precious Metals  \n\uf0b7 Metals  \n\uf0b7 Compounded Rubber  \n\uf0b7 Cigarettes  \n  \nDirect Taxes 30 \n\uf0b7 MSMEs and Professionals   \n\uf0b7 Cooperation  \n\uf0b7 Start-Ups  \n\uf0b7 Appeals  \n\uf0b7 Better tar

In [27]:
OpenAI_API_KEY=""

In [20]:
cassio_kwargs = {
    'token': ASTRA_DB_APPLICATION_TOKEN,
    'keyspace': ASTRA_DB_KEYSPACE,
}

if ASTRA_DB_SECURE_BUNDLE_PATH:
    cassio_kwargs['secure_connect_bundle'] = str(Path(ASTRA_DB_SECURE_BUNDLE_PATH).expanduser())
else:
    cassio_kwargs['database_id'] = ASTRA_DB_ID

CASSIO_READY = False
try:
    cassio.init(**cassio_kwargs)
    CASSIO_READY = True
    print(f'CassIO initialized successfully for {ASTRA_DB_ID or "secure bundle"} / {ASTRA_DB_KEYSPACE}.')
except Exception as e:
    print('CassIO initialization failed.')
    print('Use a valid Astra token with Database Administrator permissions for this DB, or set ASTRA_DB_SECURE_BUNDLE_PATH to a valid secure-connect bundle zip.')
    print(f'Details: {e}')

CassIO initialization failed.
Use a valid Astra token with Database Administrator permissions for this DB, or set ASTRA_DB_SECURE_BUNDLE_PATH to a valid secure-connect bundle zip.
Details: Generic error when fetching the URL to the secure-bundle.


In [21]:
llm=OpenAI(openai_api_key=OpenAI_API_KEY, temperature=0.5)
embedding=OpenAIEmbeddings(openai_api_key=OpenAI_API_KEY)

NameError: name 'OpenAI_API_KEY' is not defined

vectorstore 

In [22]:
astra_vectorstore=Cassandra(
       embedding=embedding,
       table_name='langchain_vectors',
       
)

NameError: name 'embedding' is not defined

In [41]:
from langchain_text_splitters import CharacterTextSplitter

In [43]:
from langchain_text_splitters import CharacterTextSplitter

text_splitter = CharacterTextSplitter(
    separator="\n",
    chunk_size=1000,
    chunk_overlap=200,
    length_function=len
)

texts = text_splitter.split_text(raw_text)
print(texts[:2])

['GOVERNMENT OF INDIA\nBUDGET 2023-2024\nSPEECH\nOF\nNIRMALA SITHARAMAN\nMINISTER OF FINANCE\nFebruary 1,  2023CONTENTS \nPART-A \n Page No. \n\uf0b7 Introduction 1 \n\uf0b7 Achievements since 2014: Leaving no one behind 2 \n\uf0b7 Vision for Amrit Kaal – an empowered and inclusive economy 3 \n\uf0b7 Priorities of this Budget 5 \ni. Inclusive Development  \nii. Reaching the Last Mile \niii. Infrastructure and Investment \niv. Unleashing the Potential \nv. Green Growth \nvi. Youth Power  \nvii. Financial Sector \n \n \n \n \n \n \n \n \n \n\uf0b7 Fiscal Management \n24 \nPART B \n  \nIndirect Taxes 27 \n\uf0b7 Green Mobility  \n\uf0b7 Electronics   \n\uf0b7 Electrical   \n\uf0b7 Chemicals and Petrochemicals   \n\uf0b7 Marine products  \n\uf0b7 Lab Grown Diamonds  \n\uf0b7 Precious Metals  \n\uf0b7 Metals  \n\uf0b7 Compounded Rubber  \n\uf0b7 Cigarettes  \n  \nDirect Taxes 30 \n\uf0b7 MSMEs and Professionals   \n\uf0b7 Cooperation  \n\uf0b7 Start-Ups  \n\uf0b7 Appeals  \n\uf0b7 Better ta

In [44]:
texts[:50]

['GOVERNMENT OF INDIA\nBUDGET 2023-2024\nSPEECH\nOF\nNIRMALA SITHARAMAN\nMINISTER OF FINANCE\nFebruary 1,  2023CONTENTS \nPART-A \n Page No. \n\uf0b7 Introduction 1 \n\uf0b7 Achievements since 2014: Leaving no one behind 2 \n\uf0b7 Vision for Amrit Kaal – an empowered and inclusive economy 3 \n\uf0b7 Priorities of this Budget 5 \ni. Inclusive Development  \nii. Reaching the Last Mile \niii. Infrastructure and Investment \niv. Unleashing the Potential \nv. Green Growth \nvi. Youth Power  \nvii. Financial Sector \n \n \n \n \n \n \n \n \n \n\uf0b7 Fiscal Management \n24 \nPART B \n  \nIndirect Taxes 27 \n\uf0b7 Green Mobility  \n\uf0b7 Electronics   \n\uf0b7 Electrical   \n\uf0b7 Chemicals and Petrochemicals   \n\uf0b7 Marine products  \n\uf0b7 Lab Grown Diamonds  \n\uf0b7 Precious Metals  \n\uf0b7 Metals  \n\uf0b7 Compounded Rubber  \n\uf0b7 Cigarettes  \n  \nDirect Taxes 30 \n\uf0b7 MSMEs and Professionals   \n\uf0b7 Cooperation  \n\uf0b7 Start-Ups  \n\uf0b7 Appeals  \n\uf0b7 Better ta

In [ ]:
texts[:50]

load the daatset into vector store 

In [ ]:
astra_vectorstore.add_texts(texts[:50])
print("Inserted %i headlines into Astra DB vector store" % len(texts[:50]))
astra_vector_index=VectorStoreIndexWrapper(vectorstore=astra_vectorstore)


In [ ]:
first_question=True 
while True:
    if first_question:
        query="What are the main topics covered in the speech?"
        first_question=False
    else:
        query=input("Ask a question about the speech (or 'exit' to quit): ")
        if query.lower() == 'exit':
            print("Exiting.")
            break

    relevant_docs = astra_vector_index.similarity_search(query, k=3)
    print("\nRelevant excerpts from the speech:")
    for i, doc in enumerate(relevant_docs):
        print(f"{i+1}. {doc.page_content[:200]}...")

the end 